Use this script to tabulate simulation status for models in this batch.

In [2]:
from pathlib import Path
from tqdm import tqdm
import pandas
import re
import sqlite3

In [3]:

def gather_batch_stats(
        root='.',
        ENERGYPLUS_MODEL_FILEPATTERN = 'instance.idf',
        ENERGYPLUS_OUT_FILEPATTERN = 'instance-out.sql',
        ENERGYPLUS_ERR_FILEPATTERN = 'instance-out.err'
):
    root = Path(root)
    result = []
    # Loop over subfolders and create progress bars.
    for subroot in root.glob('*'):
        subroot_relpath = subroot.relative_to(root)
        # Count composed models.
        subroot_composed_models = list(subroot.glob('**/runs/**/'+ENERGYPLUS_MODEL_FILEPATTERN))
        n_models = len(subroot_composed_models)

        if n_models > 0:
            # Create the progress bar here.
            myiter = tqdm(subroot_composed_models, desc=subroot_relpath.parts[0])

            for i,f1 in enumerate(myiter):
                fileerr = f1.parent.joinpath(ENERGYPLUS_ERR_FILEPATTERN)
                fileout = f1.parent.joinpath(ENERGYPLUS_OUT_FILEPATTERN)

                # Parse the filename following DEER conventions
                # In 2023, residential models would be found like this:
                # relpath = "SFm_Furnace_1975\runs\CZ01\SFm&1&rDXGF&Ex&SpaceHtg_eq__GasFurnace\Msr-Res-GasFurnace-AFUE95-ECM\instance-out.sql
                # In 2024, commercial models would be found like this:
                # relpath = "SWXX000-00 Measure Name_1975\runs\CZ01\Asm\defaults\instance-out.sql"
                relpath = f1.relative_to(root)

                meas_group_vintage_combo, _, cz, cohort, techid, _ = relpath.parts
                cohort_clean = cohort.replace("&","=")
                meas_group, bldgvint = meas_group_vintage_combo.rsplit("_", 1)

                fileerr_exists = fileerr.exists()
                fileout_exists = fileout.exists()

                if not fileerr_exists:
                    success = False
                    timestamp = None
                    warnings = None
                    errors = None
                else:
                    # Match for a line at the beginning of the file like this:
                    # Program Version,EnergyPlus, Version 9.5.0-de239b2e5f, YMD=2025.07.29 19:21,
                    content = fileerr.read_text().splitlines()
                    firstline = content[0]
                    m = re.search(r'YMD=(?P<timestamp>[^,]*)',firstline)
                    if not m:
                        timestamp = None
                    else:
                        timestamp = m['timestamp']

                    # Match for a line at the end of the file like this:
                    # ************* EnergyPlus Completed Successfully-- 4 Warning; 0 Severe Errors; Elapsed Time=00hr 01min 53.20sec
                    lastline = content[-1]
                    success = (lastline.find(r'************* EnergyPlus Completed Successfully') > 0)
                    m = re.search(r'-- (?P<warnings>\d*) Warning; (?P<errors>\d*) Severe Errors; Elapsed Time', lastline)
                    if not m:
                        warnings = None
                        errors = None
                    else:
                        warnings = m['warnings']
                        errors = m['errors']

                if not fileout_exists:
                    fileout
                else:
                    pass

                hastempdir = False
                for f3 in f1.parent.glob("instance*"):
                    if f3.is_dir():
                        hastempdir = True

                result.append((relpath, meas_group_vintage_combo, cz, cohort, techid,
                            fileerr_exists, fileout_exists,
                            timestamp, success, warnings, errors, hastempdir))

            myiter.close()
        # end models
    # end subroot folder
    stats = pandas.DataFrame(result,
                            columns=['relpath', 'meas_group_vintage_combo', 'cz', 'cohort', 'techid',
                                    'fileerr_exists', 'fileout_exists',
                                    'timestamp', 'success', 'warnings', 'errors', 'hastempdir'])
    return stats

In [4]:

stats1 = gather_batch_stats(
    root='.',
    ENERGYPLUS_MODEL_FILEPATTERN = 'instance.idf',
    ENERGYPLUS_OUT_FILEPATTERN = 'instance-out.sql',
    ENERGYPLUS_ERR_FILEPATTERN = 'instance-out.err')
stats2 = gather_batch_stats(
    root='.',
    ENERGYPLUS_MODEL_FILEPATTERN = 'instance-hardsize.idf',
    ENERGYPLUS_OUT_FILEPATTERN = 'instance-hardsize-out.sql',
    ENERGYPLUS_ERR_FILEPATTERN = 'instance-hardsize-out.err')


SWHC012-06 Occupancy Sensor_Ex:   0%|          | 0/12 [00:00<?, ?it/s]

SWHC012-06 Occupancy Sensor_New: 100%|██████████| 6/6 [00:00<00:00, 203.14it/s]


In [5]:
stats = pandas.concat([stats1, stats2])
stats

,relpath,meas_group_vintage_combo,cz,cohort,techid,fileerr_exists,fileout_exists,timestamp,success,warnings,errors,hastempdir
0,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\EPr&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,EPr&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
1,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\EPr&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,EPr&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF-NoOccSens,True,True,2026.06.13 17:21,True,3676985,0,False
2,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\EPr&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,EPr&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False
3,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\EPr&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,EPr&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP-NoOccSens,True,True,2026.06.13 17:21,True,3678369,0,False
4,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ERC&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ERC&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
5,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ERC&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ERC&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF-NoOccSens,True,True,2026.06.13 17:21,True,431627,0,False
6,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ERC&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ERC&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False
7,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ERC&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ERC&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP-NoOccSens,True,True,2026.06.13 17:21,True,431662,0,False
8,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ESe&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ESe&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
9,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ESe&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ESe&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF-NoOccSens,True,True,2026.06.13 17:21,True,15543355,0,False


In [6]:
stats[stats['success']==False]
# Note that files like 'instance.idf' in the measure case will not run through simulation. Ignore "failed" runs with that filename.
# For measure case, files like 'instance-hardsize.idf' should be simulated without errors. (Confirm in simstats.csv.)

,relpath,meas_group_vintage_combo,cz,cohort,techid,fileerr_exists,fileout_exists,timestamp,success,warnings,errors,hastempdir
0,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\EPr&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,EPr&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
2,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\EPr&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,EPr&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False
4,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ERC&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ERC&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
6,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ERC&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ERC&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False
8,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ESe&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ESe&0&cDXGF&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
10,SWHC012-06 Occupancy Sensor_Ex\runs\CZ08\ESe&0...,SWHC012-06 Occupancy Sensor_Ex,CZ08,ESe&0&cDXHP&Ex&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False
12,SWHC012-06 Occupancy Sensor_New\runs\CZ08\EPr&...,SWHC012-06 Occupancy Sensor_New,CZ08,EPr&0&cDXGF&New&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
14,SWHC012-06 Occupancy Sensor_New\runs\CZ08\EPr&...,SWHC012-06 Occupancy Sensor_New,CZ08,EPr&0&cDXHP&New&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False
16,SWHC012-06 Occupancy Sensor_New\runs\CZ08\ERC&...,SWHC012-06 Occupancy Sensor_New,CZ08,ERC&0&cDXGF&New&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXGF,False,False,None,False,None,None,False
18,SWHC012-06 Occupancy Sensor_New\runs\CZ08\ERC&...,SWHC012-06 Occupancy Sensor_New,CZ08,ERC&0&cDXHP&New&HV_Tech__OccSens,NE-HV_Tech-OccSens-cDXHP,False,False,None,False,None,None,False


In [7]:
stats[stats['hastempdir']==True]

,relpath,meas_group_vintage_combo,cz,cohort,techid,fileerr_exists,fileout_exists,timestamp,success,warnings,errors,hastempdir


In [8]:
stats.to_csv('simulation_stats.csv')


In [10]:
!simulation_stats.csv